In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def prepare_data():
    spark = (
        SparkSession.builder
        .master("local[2]")
        .appName("PySpark Interview - Order Events")
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
    )

    order_events = spark.createDataFrame(
        [
            (1, "A", "C1", "2026-10-01", "2026-10-01 10:00:00", "pending", 30),
            (2, "A", "C1", "2026-10-01", "2026-10-01 10:00:00", "completed", 35),
            (3, "B", "C2", "2026-10-01", "2026-10-01 11:00:00", "completed", 20),
            (4, "C", "C1", "2026-10-01", "2026-10-01 12:00:00", "completed", 15),
            (5, "D", "C2", "2026-10-01", "2026-10-01 13:00:00", "cancelled", 100),
            (6, "E", "C3", "2026-10-01", "2026-10-01 14:00:00", "completed", None),
            (7, "F", "C2", "2026-10-01", "2026-10-01 15:00:00", "completed", 30),
        ],
        "event_id long, order_id string, customer_id string, order_date string, "
        "updated_at string, status string, amount long",
    ).withColumn("order_date", F.to_date("order_date")) \
     .withColumn("updated_at", F.to_timestamp("updated_at"))

    order_events.show(truncate=False)
    return order_events

In [ ]:
from pyspark.sql import Window


def process_data(data_frame_1):
    print("Input data:")
    data_frame_1.show(truncate=False)

    # 1. For each order_id, select the latest event by updated_at.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").desc(), F.col("event_id").desc()
    )
    current_orders = (
        data_frame_1
        .withColumn("row_number", F.row_number().over(order_window))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .cache()
    )

    try:
        print("Current order state:")
        current_orders.show(truncate=False)

        # 2. For the current state, calculate the number of orders and sum of amount per customer and day.
        daily_summary = (
            current_orders
            .filter((F.col("status") == "completed") & F.col("amount").isNotNull())
            .groupBy("order_date", "customer_id")
            .agg(
                F.count("*").alias("order_count"),
                F.sum("amount").alias("total_amount"),
            )
        )

        print("Daily summary:")
        daily_summary.show(truncate=False)

        # 3. For each day, select the two customers with the highest total amount.
        customer_window = Window.partitionBy("order_date").orderBy(
            F.col("total_amount").desc(), F.col("customer_id").asc()
        )
        top_customers = (
            daily_summary
            .withColumn("rank", F.row_number().over(customer_window))
            .filter(F.col("rank") <= 2)
            .orderBy("order_date", "rank")
        )

        print("Customer ranking:")
        top_customers.show(truncate=False)
    finally:
        current_orders.unpersist()

    # 4. Optimization (5 min)
    # The current order state is displayed and then used to calculate the daily summary.
    # How could you avoid recomputing it? Implement a solution.


def main():
    data_frame_1 = prepare_data()
    process_data(data_frame_1=data_frame_1)


if __name__ == "__main__":
    main()

DataFrame 1:
+----------+---------+---------+--------+
|id        |str      |int      |position|
+----------+---------+---------+--------+
|0x2809906b|671715435|671715435|1       |
|0xfd9e642 |265938498|265938498|2       |
|0x1c938025|479428645|479428645|3       |
|0x23a85cbc|598236348|598236348|4       |
|0x88c98ae |143431854|143431854|5       |
|0x1273938b|309564299|309564299|6       |
|0xd29fabd |220854973|220854973|7       |
|0x184afcc4|407567556|407567556|8       |
|0x2e55bdb0|777371056|777371056|9       |
|0x3876389 |59204489 |59204489 |10      |
|0x28e480e9|686063849|686063849|11      |
|0x2703889 |40908937 |40908937 |12      |
|0x3e2e282 |65200770 |65200770 |13      |
|0xe4bd48f |239850639|239850639|14      |
|0x33a8d12f|866701615|866701615|15      |
|0x26925584|647124356|647124356|16      |
|0x7d696f8 |131503864|131503864|17      |
|0x8ec33e0 |149697504|149697504|18      |
|0x50a7a4c |84572748 |84572748 |19      |
|0x17e7c157|401064279|401064279|20      |
+----------+---------